In [1]:
import copy
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Added to sys.path:/", repo_root)
from fixedincomelib import *
print("Fixed Income Library is loaded.")

Added to sys.path:/ /Users/manushshah/Desktop/FRE 9743 - Models to Markets/GitHub/FRE-GT-9743-Assignment-6
Fixed Income Library is loaded.


## Assignment Instruction:

Please complete the valuation engine for the Overnight Index Basis Swap product, along with its corresponding API for product creation.

You could follow the design pattern of RFR Swap to complete, please see detailed instructions in linear_product.py.

**File locations:**
- `ValuationEngineProductOvernightIndexBasisSwap` → `valuation_engine/` under the `yieldcurve` folder
- `qfCreateProductOvernightIndexBasisSwap` → `product/` under the `apis` folder
- registration after implementation

**Notes:**
- Only APIs are exposed to users, so each product requires a dedicated API covering: product creation, models, and the valuation report.

After finish the codes above, you should be able to have the same output like below.

### Read in Model

In [2]:
### de-serialize a pre-built model
path = 'serialized/yc_model_multi_indices_calibrated.pickle'
yc_model : Model = qfReadModelFromFile(path)
# rebuild model for special dates
value_date = qfDisplayModelValueDate(yc_model)
base_data_collection = qfGetDataCollectionFromModel(yc_model)
build_method_collection = qfGetBuildMethodCollection(yc_model)

### Create Val Param

In [3]:
fi_rfr_vp = qfCreateValuationParameters('FUNDING INDEX PARAMETER', {'Funding Index' : 'SOFR-1B-FLAT'})
vpc_rfr = qfCreateValuationParametersCollection([fi_rfr_vp])

### Build Product Overnight Index Swap

In [4]:
effective_date = '2026-02-11'
termination_date = '2029-02-12'
pay_offset = '2D'
on_index_1 = 'SOFR-1B'
on_index_2 = 'FF-1B'
spread_leg_1 = 0.002
pay_or_rec = 'receive'
notional = 1e8
accrual_peroid_1 = '3Y'
accrual_peroid_2 = '1M'
accrual_basis = 'ACT/360'
business_day_convention = 'F'
holiday_convention = 'USGS'
compounding_method = 'compound'

product_oi_basis_swap = qfCreateProductOvernightIndexBasisSwap(
    effective_date,
    termination_date,
    pay_offset,
    on_index_1,
    on_index_2,
    spread_leg_1,
    pay_or_rec,
    notional,
    accrual_peroid_1,
    accrual_peroid_2,
    accrual_basis)
qfDisplayProduct(product_oi_basis_swap)

,Name,Value
0,Product Type,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP
1,Notional,100000000.0
2,Currency,USD
3,Long Or Short,LONG
4,Effective Date,2026-02-11
5,Termination Date,2029-02-12
6,Payment Offset,2D
7,ON Index 1,SOFRON Actual/360
8,ON Index 2,FedFundsON Actual/360
9,Spread Over Leg 1,0.002


### Test Valuation

In [5]:
### test pv and cash
report = qfCreateValueReport(yc_model, product_oi_basis_swap, vpc_rfr, 'pvdetailed')
pv_base = qfCreateValueReport(yc_model, product_oi_basis_swap, vpc_rfr, 'pv')[0][1]
display(report.display())

,Currency,Type,Value
0,USD,PV,465950.534237
1,USD,CASH,0.000000


In [6]:
### test par rate
par_rate = qfCreateValueReport(yc_model, product_oi_basis_swap, vpc_rfr, 'parrateorspread')
print(f'The swap par rate is: {par_rate:.2%}.')

The swap par rate is: 0.03%.


In [7]:
### test pv01
pv01 = qfCreateValueReport(yc_model, product_oi_basis_swap, vpc_rfr, 'pv01')
print(f'The pv01 of the swap is: {pv01:.2f}.')

The pv01 of the swap is: 276096351.65.


In [8]:
### test cf report
cf_report = qfCreateValueReport(yc_model, product_oi_basis_swap, vpc_rfr, 'cashflowsreport')
cf_report.display().T

,0,1,2,3,4,5,6,7,8,9,...,29,30,31,32,33,34,35,36,37,38
PRODUCT_TYPE,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP,...,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP
VALUATION_ENGINE_TYPE,ValuationEngineProductOvernightIndexBasisSwap,ValuationEngineProductOvernightIndexBasisSwap,ValuationEngineProductOvernightIndexBasisSwap,ValuationEngineProductOvernightIndexBasisSwap,ValuationEngineProductOvernightIndexBasisSwap,ValuationEngineProductOvernightIndexBasisSwap,ValuationEngineProductOvernightIndexBasisSwap,ValuationEngineProductOvernightIndexBasisSwap,ValuationEngineProductOvernightIndexBasisSwap,ValuationEngineProductOvernightIndexBasisSwap,...,ValuationEngineProductOvernightIndexBasisSwap,ValuationEngineProductOvernightIndexBasisSwap,ValuationEngineProductOvernightIndexBasisSwap,ValuationEngineProductOvernightIndexBasisSwap,ValuationEngineProductOvernightIndexBasisSwap,ValuationEngineProductOvernightIndexBasisSwap,ValuationEngineProductOvernightIndexBasisSwap,ValuationEngineProductOvernightIndexBasisSwap,ValuationEngineProductOvernightIndexBasisSwap,ValuationEngineProductOvernightIndexBasisSwap
LEG_ID,0,0,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
CASHFLOW_ID,0,1,0,1,2,3,4,5,6,7,...,27,28,29,30,31,32,33,34,35,36
PAY_OR_RECEIVE,1,1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
NOTIONAL,100000000.0,100000000.0,100000000.0,100000000.0,100000000.0,100000000.0,100000000.0,100000000.0,100000000.0,100000000.0,...,100000000.0,100000000.0,100000000.0,100000000.0,100000000.0,100000000.0,100000000.0,100000000.0,100000000.0,100000000.0
PAY_DATE,"February 17th, 2026","February 14th, 2029","February 17th, 2026","March 16th, 2026","April 14th, 2026","May 14th, 2026","June 16th, 2026","July 14th, 2026","August 14th, 2026","September 15th, 2026",...,"May 16th, 2028","June 14th, 2028","July 14th, 2028","August 15th, 2028","September 14th, 2028","October 16th, 2028","November 14th, 2028","December 14th, 2028","January 17th, 2029","February 14th, 2029"
FORECASTED_AMOUNT,10400.699686,10955768.999071,9330.331433,261578.614583,299002.802214,270933.352168,289645.445902,289645.445902,280288.96258,308361.0316,...,276297.723007,285520.782937,276297.723007,303969.447788,267075.511304,276297.723007,294744.691171,267075.511304,285798.751335,286304.150397
PV,10394.558019,9925641.387864,9324.821824,260730.193187,297183.34049,268490.564259,286102.961205,285368.523323,275369.831169,302065.555523,...,256697.689479,264578.067845,255335.927158,280091.376156,245424.607148,253151.771798,269330.829468,243371.900492,259607.158018,259384.103931
DISCOUNG FACTOR,0.999409,0.905974,0.999409,0.996757,0.993915,0.990984,0.98777,0.985234,0.98245,0.979584,...,0.929062,0.926651,0.924133,0.921446,0.918933,0.916228,0.913777,0.911248,0.908357,0.905974


In [9]:
### test risk
risk = qfCreateValueReport(yc_model, product_oi_basis_swap, vpc_rfr, 'firstorderrisk')
df_risk = risk.display()
df_risk.VALUES = df_risk.VALUES.round(12)
display(df_risk)

,DATA_TYPE,DATA_CONVENTION,AXIS1,AXIS2,MARKET_QUOTE,UNIT,VALUES
0,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,1Y,,0.0,0.0001,133.919376
1,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,5Y,,0.0,0.0001,-1680.441844
2,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,10Y,,0.0,0.0001,0.000000
3,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,20Y,,0.0,0.0001,0.000000
4,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,30Y,,0.0,0.0001,0.000000
5,SPREAD ZERO RATE,FF-1B-FLAT-OVER-FF-1B-ZERO-SPREAD,1Y,,0.0002,0.0001,0.000000
6,SPREAD ZERO RATE,FF-1B-FLAT-OVER-FF-1B-ZERO-SPREAD,5Y,,0.0003,0.0001,0.000000
7,SPREAD ZERO RATE,FF-1B-FLAT-OVER-FF-1B-ZERO-SPREAD,10Y,,0.0003,0.0001,0.000000
8,SPREAD ZERO RATE,FF-1B-FLAT-OVER-FF-1B-ZERO-SPREAD,20Y,,0.0003,0.0001,0.000000
9,SPREAD ZERO RATE,FF-1B-FLAT-OVER-FF-1B-ZERO-SPREAD,30Y,,0.0004,0.0001,0.000000


### Risk Validation

In [10]:
# check future
bump_size = 1e-4
data_collection_perturbed = copy.deepcopy(qfGetDataCollectionFromModel(yc_model))
# Step 1: build bumped up model
risk_tenor = '2027-12-15x2028-03-15'
dt, dconv = 'Overnight Index Future', 'SOFR-FUTURE-3M'
df = base_data_collection.get_data_from_data_collection(dt, dconv).display().copy()
df.set_index('axis1', inplace=True)
if 'FUTURE' in dt.upper():
    df.loc[risk_tenor] += bump_size / -0.01
else:
    df.loc[risk_tenor] += bump_size
data_bumped = qfCreateDataCollection([qfCreateData1D(dt, dconv, df)])
data_collection_perturbed.modify_data_collection(data_bumped)
yc_model_bumped = qfCreateModel(value_date, 'YIELD_CURVE', data_collection_perturbed, build_method_collection)
# Step 2: re-value the same product
pv_bumped = qfCreateValueReport(yc_model_bumped, product_oi_basis_swap, vpc_rfr, 'pv')[0][1]
# Step 3 : change of value due to change of rate by 1 bp
risk_bump_reval = pv_bumped - pv_base
print(f'Bump reval risk is {risk_bump_reval}.')
# Step 4: bump reval risk - analytic risk

Bump reval risk is -13.021511999890208.


In [11]:
# basis swap
bump_size = 1e-4
data_collection_perturbed = copy.deepcopy(qfGetDataCollectionFromModel(yc_model))
# Step 1: build bumped up model
risk_tenor = '3Y'
dt, dconv = 'OVERNIGHT INDEX BASIS SWAP', 'USD-FF-3M-OVER-USD-SOFR-OIS-3M'
df = base_data_collection.get_data_from_data_collection(dt, dconv).display().copy()
df.set_index('axis1', inplace=True)
if 'FUTURE' in dt.upper():
    df.loc[risk_tenor] += bump_size / -0.01
else:
    df.loc[risk_tenor] += bump_size
data_bumped = qfCreateDataCollection([qfCreateData1D(dt, dconv, df)])
data_collection_perturbed.modify_data_collection(data_bumped)
yc_model_bumped = qfCreateModel(value_date, 'YIELD_CURVE', data_collection_perturbed, build_method_collection)
# Step 2: re-value the same product
pv_bumped = qfCreateValueReport(yc_model_bumped, product_oi_basis_swap, vpc_rfr, 'pv')[0][1]
# Step 3 : change of value due to change of rate by 1 bp
risk_bump_reval = pv_bumped - pv_base
print(f'Bump reval risk is {risk_bump_reval}.')
# Step 4: bump reval risk - analytic risk

Bump reval risk is -28804.098346997052.


In [12]:
# spread zero rate
bump_size = 1e-4
data_collection_perturbed = copy.deepcopy(qfGetDataCollectionFromModel(yc_model))
# Step 1: build bumped up model
risk_tenor = '5Y'
dt, dconv = 'SPREAD ZERO RATE', 'SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD'
df = base_data_collection.get_data_from_data_collection(dt, dconv).display().copy()
df.set_index('axis1', inplace=True)
if 'FUTURE' in dt.upper():
    df.loc[risk_tenor] += bump_size / -0.01
else:
    df.loc[risk_tenor] += bump_size
data_bumped = qfCreateDataCollection([qfCreateData1D(dt, dconv, df)])
data_collection_perturbed.modify_data_collection(data_bumped)
yc_model_bumped = qfCreateModel(value_date, 'YIELD_CURVE', data_collection_perturbed, build_method_collection)
# Step 2: re-value the same product
pv_bumped = qfCreateValueReport(yc_model_bumped, product_oi_basis_swap, vpc_rfr, 'pv')[0][1]
# Step 3 : change of value due to change of rate by 1 bp
risk_bump_reval = pv_bumped - pv_base
print(f'Bump reval risk is {risk_bump_reval}.')
# Step 4: bump reval risk - analytic risk

Bump reval risk is -1680.1986775975674.
